In [12]:
import librosa
import numpy as np
import matplotlib as plt

First why we go for chromagrams instead of spectrograms?

What we are looking for in this task is to look for the notes or rhythm regradless of the instrument being played. Spectrograms give us frequency info which is dependent on instrument. On the other hand, chromagrams give us the note beeing played regardless of the instrument. Hence chromagrams are a better choice for this task.

And of course all comes at cost of computations.

Also it would be benefical to know when to use CQT ( Constant Q Transform ) for chromagrams or STFT : 

|  FFT / STFT  |  CQT  |
|--------------------|--------------|
|  care about **precise timing** |  care about **pitch and harmony** "Our Case"|
|  need **phase information** | analyzing **music**  "Our Case"|
| **speech or general DSP** | feeding audio into **ML models** |
| prefer  **speed** | **octave-invariant features** |

A high level comparison between CQT and FFT [
Fast Fourier Transform and Constant Q Explanation](https://youtu.be/Cl-m4X3rwac)

In [17]:
# O( N × M ) For a big database like shazam it will take too long , but for now let's stick for the idea

def extract_features(audio_path):
    """
    Load audio -> extract Chroma features with CQT -
    """
    # Load audio
    y, sr=librosa.load(audio_path, sr=None)

    # Extract Chroma CQT features ( all the 12 notes * time)
    # Foucsing only on the harmony and ignoring variances

    chroma=librosa.feature.chroma_cqt(y=y, sr=sr)

    return chroma

def compute_dtw_distance(humming_chroma, song_chroma):
    """
    Similar to Edit Disntace problem. We need the 
    "Cheapest Path" that aligns with the input record with the real chroma of the song
    """

    # Use of cosine for similarity via dot products
    D,wp=librosa.sequence.dtw(humming_chroma,song_chroma,metric='cosine')

    # The value we need at the corner of the matrix ( the total ditance )
    return D[-1,-1]/len(wp)

def match(humming_path,song_path):
    humming_chroma=extract_features(humming_path)
    song_chroma=extract_features(song_path)

    # Similar to djkstra algorithm , we need the shortest path that aligns 
    # the user input to and compare 

    best_distance=float('inf')
    best_shift=0

    # Since we have only 12 note, just brute force all of them
    for i in range(12):
        # Shift notes
        shifted_humming=np.roll(humming_chroma,i,axis=0)

        # Calculate distance
        distance=compute_dtw_distance(shifted_humming,song_chroma)

        if distance<best_distance:
            best_distance=distance
            best_shift=i
    print(f"best match distance: {best_distance} with shift of {best_shift} semitones")
    return best_distance

In [24]:
match(r'Tests\2\الشبيه.mp3', r'Tests\2\Aaron Smith - Dancin (KRONO Remix)(M4A_128K).m4a')
# The lower the better match we got

C:\Users\pcc\AppData\Local\Temp\ipykernel_19888\2140103972.py:8: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr=librosa.load(audio_path, sr=None)


best match distance: 0.06967222015620186 with shift of 10 semitones


np.float64(0.06967222015620186)

To generate audio embeddings, several steps are typically involved. Initially, audio data is pre-processed to ensure consistency and quality. This may include operations such as resampling, normalization, and noise reduction. Once pre-processed, the audio signal is often divided into smaller segments or frames to facilitate detailed analysis over time.

then it follow the extraction phase . Common ones
1. Mel Frequency Cepstral Coefficients (MFCCs): These coefficients capture the short-term power spectrum of sound and are widely used in speech and audio processing tasks.
2. Chroma Features: These features represent the intensity of each of the 12 distinct semitones (or chroma) of the musical octave, providing insights into the harmonic content of the audio.
3. Spectrogram Features: Spectrograms visualize the frequency spectrum of the audio signal over time, allowing for the extraction of various statistical features.

Then being fed to a model usuall CNN or RNN to learn patterns and generate embeddings due to ability to capture temporal and spectral features of audio data.

Finally, the model proccess it  and outputs  a vector of fixed length which is the audio embedding representing the audio clip in a lower dimensional space while preserving its essential characteristics.